# ParsingAI — Colab 版
依序執行每個 Cell。

In [ ]:
# ── Cell 1: 安裝套件 & 設定 ────────────────────────────────────────────────────
!git clone https://github.com/410773004/parsingAI.git /content/parsingAI -q
!pip install ollama tiktoken -q
!apt-get install -y zstd pciutils -q
!curl -fsSL https://ollama.com/install.sh | sh

import sys, subprocess, time
sys.path.insert(0, '/content/parsingAI')
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

MODEL = 'qwen3:4b'
print(f'拉取模型：{MODEL}')
!ollama pull {MODEL}
print('✓ 完成')

In [ ]:
# ── Cell 2: 上傳處理過的 log txt ───────────────────────────────────────────────
import re, sys
if '/content/parsingAI' not in sys.path:
    sys.path.insert(0, '/content/parsingAI')

from google.colab import files
from pathlib import Path
from parsers.compress import count_tokens

uploaded = files.upload()
log_filename = list(uploaded.keys())[0]
cleaned_log = Path(log_filename).read_text(encoding='utf-8', errors='ignore')

# 從 log 自動抓 project / fw_version
fw_match = re.search(r'Firmware Version\s*:\s*(\S+)', cleaned_log)
fw_version = fw_match.group(1) if fw_match else ''

if 'LJ1(SSSTC)' in cleaned_log or re.search(r'FG\d+', cleaned_log):
    project = 'PJ1'
elif 'SSSTC ER3' in cleaned_log or 'FWInfo' in cleaned_log:
    project = 'ER3'
else:
    project = 'PJ1'

print(f'檔案      : {log_filename}')
print(f'Project   : {project}')
print(f'FW Version: {fw_version}')
print(f'Token 數  : {count_tokens(cleaned_log):,}')

In [ ]:
# ── Cell 3: 執行 LLM 分析 ──────────────────────────────────────────────────────
# 只需修改這行
issue = '請輸入問題描述'

import sys, subprocess, time, httpx, math, importlib
if '/content/parsingAI' not in sys.path:
    sys.path.insert(0, '/content/parsingAI')

try:
    httpx.get('http://127.0.0.1:11434', timeout=2)
except Exception:
    print('啟動 Ollama server...')
    subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)

import ollama
import settings.prompts as _prompts_mod
importlib.reload(_prompts_mod)
from settings.prompts import SYSTEM, USER_TEMPLATE
from parsers.compress import count_tokens

_log_tokens = count_tokens(cleaned_log)
_num_ctx = math.ceil((_log_tokens + 1500) / 1000) * 1000
print(f'log tokens: {_log_tokens:,}  →  num_ctx: {_num_ctx:,}\n')

USER_PROMPT = USER_TEMPLATE.format(
    model=project,
    fw_version=fw_version,
    issue=issue,
    log_text=cleaned_log,
)

full_text = ''
for chunk in ollama.chat(
    model=MODEL,
    messages=[
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user',   'content': USER_PROMPT},
    ],
    options={'num_ctx': _num_ctx},
    stream=True,
):
    token = chunk.get('message', {}).get('content', '')
    full_text += token
    print(token, end='', flush=True)